[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/adairgranados/adair-granados?quickstart=1)


<!-- gen:values.yml#weeks.07.notebook-title|Demonstration -->
# Week 7 Demonstration: Independent practice week
<!-- /gen -->

*A worked audit, then <!-- plain-language: everyday decision -->decisions, then a clean pipeline, on two made-up tables with the fire-brigade tables' problems. The <!-- plain-language: everyday decision -->decision log is written as your team's record should be written ... let's go!*

## How to use this notebook

This notebook runs your group project's cleaning pipeline on two made-up tables. They contain six months of 999 calls to Ashdown Vale Fire and Rescue, a fictional six-station service. The notebook generates them itself, from the module seed `5813` (data generation). They use the real tables' column names and have the same problems, so every step works on the real tables without renaming a column.

| Problem | In the Ashdown Vale tables | In the fire-brigade tables |
|---------|-----------------|----------------------------|
| Engines whose incident is not in the table (the join gap) | 18 of 900 engine rows carry incident numbers from December, before the incidents table starts | About 1,400 of some 336,000 engine rows, because the two extracts are cut on different days: Challenge 1 in the [data note](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/lfb-data.md) |
| Attendance time empty where no engine attended | 30 incidents with `NumPumpsAttending` 0 and a blank `FirstPumpArriving_AttendanceTime` | About one incident in twenty: Challenge 2 in the data note |
| Timestamps in mixed formats, with blanks | `DateAndTimeReturned` in two formats and blank on four rows in ten | `DateAndTimeReturned` missing on many rows and in mixed formats. Every `DateAndTime*` column arrives as text |
| Place names in mixed case | `IncGeo_BoroughName` in three cases | Capitals in both tables in the current extract, ProperCase elsewhere in the Brigade's files: Check before you join or group on a name |
| Repeated engine rows | 9 mobilisation rows duplicated wholesale, `ResourceMobilisationId` included | A couple of thousand repeated `ResourceMobilisationId`s in the extract this notebook was written against |
| A category in two spellings | `IncidentGroup` as `False Alarm` and `False alarm` | `IncidentGroup` is consistent in the current extract (blank on a handful of rows) |

Count the repeated ids in your own copy, and run `value_counts()` on every category you group by.

Between the code cells are [`DECISION:`](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-decision-entry) entries. In your [Sprint Zero](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-sprint-zero), the <!-- plain-language: everyday decision -->decision goes into your team's record first, and then the notebook implements it. The [Stoa workflow guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/stoa-workflow.md), section 4, explains the convention. Parts 1 and 2 each include an [AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens).

## Setup: Generate the two messy tables

The cell below generates Ashdown Vale's two tables (600 incidents and 900 engine rows) and writes them to CSV.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import os
import numpy as np
import pandas as pd

DATA_DIR = "assets/data/week07"
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(5813)  # module seed: Do not change

# --- Stations, districts, and each station's home district.
stations = ["Netherwick", "Ashby Mere", "Haverling", "Pendlebury Cross", "Wexcombe", "Sallowford"]
districts = ["NORTHGATE", "RIVERSIDE", "DOWNLANDS", "OLD TOWN"]
home = {"Netherwick": "NORTHGATE", "Ashby Mere": "NORTHGATE", "Haverling": "RIVERSIDE",
        "Pendlebury Cross": "DOWNLANDS", "Wexcombe": "DOWNLANDS", "Sallowford": "OLD TOWN"}

# --- 600 incidents, 1 January to 30 June 2026. 570 had an engine arrive.
# --- 30 were stood down or handled without one (NumPumpsAttending 0).
n = 600
call = (pd.Timestamp("2026-01-01")
        + pd.to_timedelta(rng.integers(0, 181, size=n), unit="D")
        + pd.to_timedelta(rng.integers(0, 86400, size=n), unit="s")).sort_values()
pumps = np.array([1] * 327 + [2] * 189 + [3] * 48 + [4] * 6 + [0] * 30)
rng.shuffle(pumps)
station = rng.choice(stations, size=n)
inc = pd.DataFrame({
    "IncidentNumber": [f"{i:06d}-{t:%d%m%Y}" for i, t in enumerate(call, start=1)],
    "DateOfCall": call.strftime("%Y-%m-%d"),
    "TimeOfCall": call.strftime("%H:%M:%S"),
    "HourOfCall": call.hour,
    "IncidentGroup": rng.choice(["False Alarm", "Special Service", "Fire"], size=n, p=[0.45, 0.40, 0.15]),
    "PropertyCategory": rng.choice(["Dwelling", "Non Residential", "Outdoor", "Road Vehicle"],
                                   size=n, p=[0.6, 0.15, 0.15, 0.1]),
    "IncGeo_BoroughName": [home[s] for s in station],
    "IncidentStationGround": station,
    "NumPumpsAttending": pumps,
})
first = np.clip(rng.normal(320, 90, size=n), 90, 1200).round()
first = first + np.where(inc["IncGeo_BoroughName"] == "DOWNLANDS", 75, 0)  # one slow district
inc["FirstPumpArriving_AttendanceTime"] = np.where(pumps > 0, first, np.nan)
inc["PumpMinutesRounded"] = np.maximum(pumps, 1) * rng.choice([60, 60, 60, 120, 180], size=n)
inc["Notional Cost (£)"] = (inc["PumpMinutesRounded"] / 60 * 467).round().astype(int)

# --- One mobilisation row per engine that attended: 873 rows.
rows = inc.loc[inc.index.repeat(inc["NumPumpsAttending"])].copy()
rows["PumpOrder"] = rows.groupby("IncidentNumber").cumcount() + 1
mobilised = (pd.to_datetime(rows["DateOfCall"] + " " + rows["TimeOfCall"])
             + pd.to_timedelta(rng.integers(30, 120, size=len(rows)), unit="s"))
extra = np.where(rows["PumpOrder"] == 1, 0, rng.integers(30, 400, size=len(rows)))
attendance = rows["FirstPumpArriving_AttendanceTime"].to_numpy() + extra
arrived = mobilised + pd.to_timedelta(attendance, unit="s")
returned = arrived + pd.to_timedelta(rng.integers(20, 90, size=len(rows)), unit="m")
mob = pd.DataFrame({
    "ResourceMobilisationId": np.arange(500001, 500001 + len(rows)),
    "IncidentNumber": rows["IncidentNumber"].to_numpy(),
    "Resource_Code": [f"{s[:2].upper()}{rng.integers(1, 3)}1" for s in rows["IncidentStationGround"]],
    "DeployedFromStation_Name": rows["IncidentStationGround"].to_numpy(),
    "PerformanceReporting": np.where(rows["PumpOrder"] == 1, "1", np.where(rows["PumpOrder"] == 2, "2", "Not Used")),
    "DateAndTimeMobilised": mobilised.dt.strftime("%d/%m/%Y %H:%M").to_numpy(),
    "DateAndTimeArrived": arrived.dt.strftime("%d/%m/%Y %H:%M").to_numpy(),
    "AttendanceTimeSeconds": attendance.astype(int),
    "DateAndTimeReturned": returned.dt.strftime("%d/%m/%Y %H:%M").to_numpy(),
    "DelayCode_Description": rng.choice([None, "Not held up", "Traffic, roadworks, etc", "Address incomplete/wrong"],
                                        size=len(rows), p=[0.7, 0.15, 0.1, 0.05]),
})

# --- Deliberate mess 1: 18 engine rows for December 2025 incidents that the
# --- incidents table does not cover (the join gap).
stray_dates = pd.Timestamp("2025-12-20") + pd.to_timedelta(rng.integers(0, 12, size=18), unit="D")
stray = mob.sample(18, random_state=5813).copy()
stray["IncidentNumber"] = [f"{i:06d}-{d:%d%m%Y}" for i, d in zip(range(9101, 9119), stray_dates)]
stray["ResourceMobilisationId"] = np.arange(499001, 499019)
stray_mobilised = stray_dates + pd.to_timedelta(rng.integers(0, 86400, size=18), unit="s")
stray["DateAndTimeMobilised"] = stray_mobilised.strftime("%d/%m/%Y %H:%M").to_numpy()
stray["DateAndTimeArrived"] = (stray_mobilised + pd.to_timedelta(stray["AttendanceTimeSeconds"].to_numpy(), unit="s")).strftime("%d/%m/%Y %H:%M").to_numpy()
stray["DateAndTimeReturned"] = None
mob = pd.concat([mob, stray], ignore_index=True)

# --- Deliberate mess 2: DateAndTimeReturned in two formats, 40% blank.
ret = pd.to_datetime(mob["DateAndTimeReturned"], format="%d/%m/%Y %H:%M", errors="coerce")
iso = rng.random(len(mob)) < 0.3
mob["DateAndTimeReturned"] = np.where(iso, ret.dt.strftime("%Y-%m-%d %H:%M:%S"), mob["DateAndTimeReturned"])
mob.loc[rng.random(len(mob)) < 0.4, "DateAndTimeReturned"] = None
mob.loc[ret.isna(), "DateAndTimeReturned"] = None

# --- Deliberate mess 3: 9 engine rows repeated wholesale.
mob = pd.concat([mob, mob.sample(9, random_state=5814)], ignore_index=True)
mob = mob.sample(frac=1, random_state=5815).reset_index(drop=True)

# --- Deliberate mess 4: Borough names in three cases.
messy = rng.choice(n, size=60, replace=False)
inc.loc[messy[:40], "IncGeo_BoroughName"] = inc.loc[messy[:40], "IncGeo_BoroughName"].str.title()
inc.loc[messy[40:], "IncGeo_BoroughName"] = inc.loc[messy[40:], "IncGeo_BoroughName"].str.lower()

# --- Deliberate mess 5: One incident category in two spellings.
false_alarm = inc.index[inc["IncidentGroup"] == "False Alarm"]
inc.loc[rng.choice(false_alarm, size=40, replace=False), "IncidentGroup"] = "False alarm"

inc.to_csv(f"{DATA_DIR}/incidents.csv", index=False)
mob.to_csv(f"{DATA_DIR}/mobilisations.csv", index=False)
print(f"Created {DATA_DIR}/incidents.csv ({len(inc)} rows) and "
      f"{DATA_DIR}/mobilisations.csv ({len(mob)} rows)")

## Part 1: The audit

The audit finds the problems in the tables before anyone changes the data. In Sprint Zero, each member audits a slice of the tables and types the findings into your team's record.

Load the two tables with [`pd.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) and look at the first rows. `dtype={"IncidentNumber": str}` keeps the incident number as text in both tables. The leading zeros survive, and the join key has the same type on both sides. For now, just look; the checks below find the problems.

In [ ]:
import pandas as pd

incidents = pd.read_csv("assets/data/week07/incidents.csv", dtype={"IncidentNumber": str})
mobilisations = pd.read_csv("assets/data/week07/mobilisations.csv", dtype={"IncidentNumber": str})
incidents.head()

The first rows look fine. [`.info()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) reports every column's type and its non-null count (the number of values that are not blank) for each table. It shows the blanks and the wrongly typed columns that `head()` does not.

In [ ]:
incidents.info()
print()
mobilisations.info()

Read the output for these findings:

- `FirstPumpArriving_AttendanceTime` has fewer non-null values than the incidents table has rows, which means blanks. The audit finds out what the blanks mean
- Every `DateAndTime*` column in the mobilisations table is text. Time calculations need them parsed first. How you parse them is a <!-- plain-language: everyday decision -->decision to log
- `PerformanceReporting` is text too, because it mixes `1`, `2`, and `Not Used`, as the real column does. A `groupby` that calculates with it fails with an error
- `DateAndTimeReturned` and `DelayCode_Description` have many blanks. Find out whether an empty delay code means *no delay* or *not recorded*, and whether the returned time is worth parsing

> 🤖 **AI Lens:** Before you audit your slice of the real tables, ask your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) the question below. Then compare its guesses with what your audit finds. Note what it predicted correctly, what it missed, and any problem it invented. Log only what your audit confirms.

```text
What data-quality problems would you expect in a fire brigade's incident and mobilisation records, where one table has a row per 999 call, the other a row per engine sent, and the two join on an incident number?
```

**A good answer** lists plausible problems. They include engines whose incident is missing from the other table, and arrival times empty where no engine arrived. They also include timestamps stored as text in more than one format, place names cased differently across the two tables, and repeated engine rows.

Next come the gaps. [`DataFrame.isna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) with `.sum()` counts the missing values in every column of each table.

In [ ]:
print(incidents.isna().sum(), "\n")
print(mobilisations.isna().sum())

The incidents table has thirty blank attendance times. Before anyone proposes filling them, check what happened on those rows. The data note explains that the blanks are calls no engine arrived at. The cell below checks this in one line, using `NumPumpsAttending`.

In [ ]:
# Challenge 2 in the data note: Engines sent where the attendance time is blank
no_time = incidents["FirstPumpArriving_AttendanceTime"].isna()
incidents.loc[no_time, "NumPumpsAttending"].value_counts()

Count the repeats two ways: Rows identical in every column, and engine rows per incident. [`.duplicated()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) marks a row that repeats an earlier one.

In [ ]:
print(f"Fully duplicated incident rows: {incidents.duplicated().sum()}")
print(f"Fully duplicated engine rows:   {mobilisations.duplicated().sum()}")
print(f"Repeated ResourceMobilisationId: {mobilisations['ResourceMobilisationId'].duplicated().sum()}")
print("\nIncidents by number of engine rows:")
print(mobilisations["IncidentNumber"].value_counts().value_counts().sort_index())

Tell these kinds of repeat apart, here and in the real tables:

- Several engine rows for one incident are normal. The **grain of the mobilisations table** is one row per engine sent, so three rows mean three engines were sent
- A row identical in **every** column, `ResourceMobilisationId` included, is a duplicate. The nine repeated ids are the nine fully duplicated rows

Category consistency comes next. [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) on `IncidentGroup` and on the borough name lists every distinct label with its count.

In [ ]:
print(incidents["IncidentGroup"].value_counts(), "\n")
print(incidents["IncGeo_BoroughName"].value_counts())

Three incident groups have become four labels, and four districts have become twelve. The labels differ only in their capital letters. A join on a name that is `LAMBETH` on one side and `Lambeth` on the other matches nothing.

The data note's first check is whether the two tables join. [`Series.isin`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) checks this in both directions. It counts engine rows whose incident number the incidents table does not have, and incidents that have no engine row at all.

In [ ]:
has_incident = mobilisations["IncidentNumber"].isin(incidents["IncidentNumber"])
has_engine = incidents["IncidentNumber"].isin(mobilisations["IncidentNumber"])
print(f"Engine rows whose incident is not in the incidents table: {(~has_incident).sum()} of {len(mobilisations)}")
print(f"Incidents with no engine row at all: {(~has_engine).sum()} of {len(incidents)}")

Eighteen engines have no incident, and thirty incidents have no engine. These are the same thirty incidents whose attendance time is blank, so the two findings are consistent. An inner join would drop the eighteen without a warning. A left join keeps them with empty incident columns. Either join is acceptable, as long as you notice what happened to the eighteen.

The last check passes the returned-time column to [`pd.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) and leaves pandas to guess its format. The cell below is **expected to fail**. Run it and read the error.

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!! EXPECTED TO FAIL !!!!!!!!!!!!!!!!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# DateAndTimeReturned mixes two formats
pd.to_datetime(mobilisations["DateAndTimeReturned"])

Here pandas guessed a format from the first value and stopped at the first value that did not fit. This error helps you, because the notebook stops. A parse that guesses wrong and carries on is worse, because it can turn 3 April into 4 March without a warning.

Type the audit findings into your team's record as concrete numbered points, with no solutions yet:

> **Audit findings (incidents.csv, 600 rows, and mobilisations.csv, 900 rows):**
> 1. 18 engine rows carry an incident number the incidents table does not have. All of them are December incidents, before the incidents table starts. 30 incidents have no engine row
> 2. `FirstPumpArriving_AttendanceTime` is blank on 30 incidents (5.0%). `NumPumpsAttending` is 0 on every one of them: No engine arrived, so there is no time to record
> 3. 9 fully duplicated engine rows, with the same 9 `ResourceMobilisationId`s repeated
> 4. `DateAndTimeReturned` is blank on 372 of 900 rows and filled in two formats (`20/01/2026 13:29` and `2026-02-10 19:50:00`). Every other `DateAndTime*` column is text in one format
> 5. `IncidentGroup` has 4 spellings of 3 groups (`False alarm`). `IncGeo_BoroughName` has 12 spellings of 4 districts, by case alone
> 6. `DelayCode_Description` is blank on about seven engine rows in ten *and* carries `Not held up` as a value. Whether a blank means the same thing is a question for the client

## Part 2: Decisions, then cleaning

Model your own cleaning section on this one. Each cleaning step opens with a `DECISION:` entry (what, why, owner), then the code that implements it, then the check that it worked.

---

> **`DECISION:`** Drop engine rows that are identical in **every** column, `ResourceMobilisationId` included. The grain of the mobilisations table is one row per engine sent, so an id that repeats is the same dispatch recorded twice. Rows that merely share an incident number stay, because that is what several engines at one incident looks like. Expected removal: 9 rows. Verify that `ResourceMobilisationId` is unique afterwards. Owner: Marta implements and confirms the count.

[`DataFrame.drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) keeps the first of any set of identical rows. The `assert` line is the verification: The notebook stops with an error if the <!-- plain-language: everyday decision -->decision's prediction is wrong.

In [ ]:
rows_before = len(mobilisations)
mobilisations = mobilisations.drop_duplicates()

print(f"Engine rows: {rows_before} -> {len(mobilisations)} "
      f"(removed {rows_before - len(mobilisations)}; decision predicted 9)")
assert mobilisations["ResourceMobilisationId"].is_unique, "every engine row should now have its own id"
print("Verified: ResourceMobilisationId is unique.")

If the first line had printed 60, the rule would be removing rows the audit never flagged. You would catch that here, before the join copies the error into more rows.

---

Teams often argue about the missing-values <!-- plain-language: everyday decision -->decision. End the argument with a logged <!-- plain-language: everyday decision -->decision, like this one:

> **`DECISION:`** `FirstPumpArriving_AttendanceTime` stays blank where no engine attended. A blank here means "no engine arrived", so filling it with a median would invent 30 response times for calls that had none. We add an `EngineAttended` flag (`NumPumpsAttending > 0`). Every response-time figure is computed over attended incidents and states this in its caption. The excluded share is reported once, in the data section. We considered filling with the median and rejected it because the invented arrivals would make every district's average look better than it is. Owner: Elena adds the flag and the caption convention. Farid documents the excluded share.

> 🤖 **AI Lens:** This <!-- plain-language: everyday decision -->decision started as a disagreement. Expect the same argument in your team, about the real column. When it comes, give the room's Associate the prompt below. Read the summary back as a team, and check it against what was said. A `DECISION:` that records the losing option as "considered and rejected because…" is strong evidence of your process.

```text
Summarise our disagreement about the empty attendance times. State each position and its strongest argument fairly, then list what evidence from our own audit would settle it.
```

**A good answer** states both positions fairly enough that each side accepts the summary of its own view. It names evidence from your own audit and leaves the choice to your team. That evidence is how many attendance times are blank, what `NumPumpsAttending` shows on those rows, and which of your questions use the column.

The cell below implements the <!-- plain-language: everyday decision -->decision. It adds the `EngineAttended` flag, checks that the flagged rows are the blank ones, and computes the mean first-engine time over attended incidents only.

In [ ]:
incidents["EngineAttended"] = incidents["NumPumpsAttending"] > 0
no_time = incidents["FirstPumpArriving_AttendanceTime"].isna()

print(f"Blank attendance times: {no_time.sum()} of {len(incidents)} incidents ({no_time.mean():.1%}), left blank")
print(f"Incidents with EngineAttended False: {(~incidents['EngineAttended']).sum()} (should be the same rows)")
assert (no_time == ~incidents["EngineAttended"]).all()

# Response time over attended incidents only, with n in the caption
attended = incidents[incidents["EngineAttended"]]
print(f"\nMean first-engine time, attended incidents only (n = {len(attended)}): "
      f"{attended['FirstPumpArriving_AttendanceTime'].mean():.0f} s")

Filling the attendance times with a median takes one line of code. It is still a large analytical choice, so it needs a `DECISION:` entry.

---

> **`DECISION:`** Parse `DateAndTimeReturned` one explicit format at a time, because a guessed format can swap the day and the month. Coerce what does not match to `NaT` (the pandas marker for a missing timestamp). Verify the result two ways: Every filled value parses, and every engine returns *after* it arrived and within three hours. The other `DateAndTime*` columns are in one format and get that format spelled out. Owner: Dayo. *(Real-table translation: The same rule applies to every timestamp column you use. If `DateAndTimeReturned` is too sparse to be worth it, log that and leave it alone.)*

The Associate's first draft for this step is below. It contains a mistake that is easy to miss. `format="mixed"` asks pandas to guess each value's format, and `dayfirst=True` looks like the right setting for British dates. Run it, and check the verification line.

In [ ]:
# The Associate's first draft: "Let pandas work out the formats"
arrived = pd.to_datetime(mobilisations["DateAndTimeArrived"], format="%d/%m/%Y %H:%M")
draft = pd.to_datetime(mobilisations["DateAndTimeReturned"], format="mixed", dayfirst=True)

turnaround = draft - arrived
implausible = (turnaround < pd.Timedelta(0)) | (turnaround > pd.Timedelta(hours=3))
print(f"Parsed: {draft.notna().sum()} of {mobilisations['DateAndTimeReturned'].notna().sum()} filled values")
print(f"Engines back at station before they arrived, or more than 3 hours after: {implausible.sum()}")

Every value parsed, but dozens of turnarounds cannot be right: Engines back at station before they reached the incident, or hours after it. The draft read the year-first values (`2026-02-10 19:50:00`) with the day and the month swapped wherever the day was twelve or less. 10 February became 2 October, and pandas gave no warning. The verification line found the problem. Reject the draft, log why, and parse explicitly: `pd.to_datetime` once per format with `errors="coerce"`, and [`Series.fillna`](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) to combine the two passes.

In [ ]:
raw = mobilisations["DateAndTimeReturned"]
returned = (pd.to_datetime(raw, format="%d/%m/%Y %H:%M", errors="coerce")
            .fillna(pd.to_datetime(raw, format="%Y-%m-%d %H:%M:%S", errors="coerce")))
mobilisations["ReturnedAt"] = returned

turnaround = returned - arrived
implausible = (turnaround < pd.Timedelta(0)) | (turnaround > pd.Timedelta(hours=3))
print(f"Parsed: {returned.notna().sum()} of {raw.notna().sum()} filled values (should match)")
print(f"Implausible turnarounds: {implausible.sum()} (decision expects 0)")
assert returned.notna().sum() == raw.notna().sum() and implausible.sum() == 0
print(f"Verified. Median time from arrival to return: {turnaround.median()}")

---

> **`DECISION:`** Normalise the names mechanically. Strip `IncGeo_BoroughName` and upper-case it, because the real tables carry boroughs in capitals and any join or group on a name happens after this step. Fix `IncidentGroup` by explicit mapping (`False alarm` → `False Alarm`), kept in the notebook where it can be reviewed and extended. Verify by re-running `value_counts()` and confirming exactly 4 districts and 3 groups remain. Owner: Ben. *(Real-table translation: The mechanical step is the same. The explicit mapping is how you would reconcile a station named one way in `IncidentStationGround` and another in `DeployedFromStation_Name`. Log each such mapping as its own entry.)*

The mechanical step uses pandas [string methods](https://pandas.pydata.org/docs/user_guide/text.html), which edit every value in a text column at once. The mapping step is [`Series.replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html), which swaps values by dictionary lookup.

In [ ]:
group_map = {"False alarm": "False Alarm"}  # Add an entry for each further label to merge

incidents["IncGeo_BoroughName"] = incidents["IncGeo_BoroughName"].str.strip().str.upper()
incidents["IncidentGroup"] = incidents["IncidentGroup"].replace(group_map)

print(incidents["IncGeo_BoroughName"].value_counts(), "\n")
print(incidents["IncidentGroup"].value_counts())
assert incidents["IncGeo_BoroughName"].nunique() == 4, "expected exactly 4 districts after cleaning"
assert incidents["IncidentGroup"].nunique() == 3, "expected exactly 3 incident groups after cleaning"
print("\nVerified: 4 districts and 3 incident groups, as decided.")

---

Join the tables last, once each one is clean, because a join copies the problems in either table into the merged table.

> **`DECISION:`** Left join mobilisations to incidents on `IncidentNumber`, so every engine row is kept. Set the 18 engine rows whose incident is missing aside in their own table. Do not drop them or leave them in with empty incident columns. They are December calls the incidents table does not cover, and a station-workload count needs every engine. Print the row counts before and after. The left join must not change the engine count. Owner: Sam writes the join cell and the counts.

The tool is [`DataFrame.merge`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) from Week 5, with `how="left"` and one addition: `indicator=True` adds a `_merge` column showing, for every row, whether it matched. The verification uses that column.

In [ ]:
rows_before = len(mobilisations)
merged = mobilisations.merge(incidents, on="IncidentNumber", how="left", indicator=True)
print(merged["_merge"].value_counts(), "\n")

engines = merged[merged["_merge"] == "both"].drop(columns="_merge")
orphans = merged[merged["_merge"] == "left_only"].drop(columns="_merge")
assert len(merged) == rows_before, "a left join must not change the engine count"
assert len(orphans) == 18, "decision predicted 18 engines with no incident"
print(f"Engine rows: {rows_before} -> {len(engines)} matched + {len(orphans)} set aside")
print("Months the set-aside incident numbers end in:", sorted(orphans["IncidentNumber"].str[-6:].unique()))

Keep the `orphans` table, so the report can state how many engine rows were set aside and why.

## Part 3: Does the cleaning pay off?

The passes below run over the cleaned tables. The first answers the response-time question from the data note. [`groupby`](https://pandas.pydata.org/docs/user_guide/groupby.html) with `agg` gives the first engine's time by district, over attended incidents only. The table also includes the share over the Brigade's six-minute target. Before the case fix, `Downlands` and `DOWNLANDS` would count as two districts. Before the attendance <!-- plain-language: everyday decision -->decision, the times would be misleading.

Here the `lambda` is a one-line unnamed function that gives each district's share of times over 360 seconds (optional background: [Week 2b](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Introduction.ipynb)).

In [ ]:
# The printed line is the table's caption
attended = incidents[incidents["EngineAttended"]]
by_district = attended.groupby("IncGeo_BoroughName")["FirstPumpArriving_AttendanceTime"]

response = by_district.agg(["count", "median"]).round(0)
response["share_over_6_min"] = by_district.apply(lambda s: (s > 360).mean()).round(2)
print(f"Attended incidents: {len(attended)} of {len(incidents)} "
      f"({(~incidents['EngineAttended']).sum()} with no engine excluded)")
response

One district stands out, and you can defend the number, because the caption states which incidents it covers. The second pass answers the false-alarm question: What share of each property category's calls are false alarms? [`pd.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) gives one row per category with the share in each group. Without the spelling fix, the shares would split across four labels.

In [ ]:
pd.crosstab(incidents["PropertyCategory"], incidents["IncidentGroup"], normalize="index").round(2)

The third pass is a cross-check. Copy it into your own notebook. The incidents table *records* how many engines attended (`NumPumpsAttending`). The join *shows* how many engine rows each incident has. If the two agree, the join is right and so is the duplicate <!-- plain-language: everyday decision -->decision. If not, an earlier step is wrong.

In [ ]:
engines_per_incident = engines.groupby("IncidentNumber").size()
claimed = incidents.set_index("IncidentNumber")["NumPumpsAttending"]
agree = engines_per_incident.reindex(claimed.index, fill_value=0) == claimed
print(f"Incidents where the engine rows match NumPumpsAttending: {agree.sum()} of {len(claimed)}")
assert agree.all(), "the join and the incidents table disagree about engine counts"

Finally, build the cleaned incidents table in the shape the [brief's section 4](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/group-project.md) asks for, and write it to its own file. The table has one row per incident, with `IncidentNumber` as the key. It contains the named columns, the flag the attendance <!-- plain-language: everyday decision -->decision added, and the engine count the join verified.

In [ ]:
keep = ["IncidentNumber", "DateOfCall", "IncidentGroup", "IncGeo_BoroughName", "IncidentStationGround",
        "FirstPumpArriving_AttendanceTime", "NumPumpsAttending", "PumpMinutesRounded", "Notional Cost (£)"]
cleaned = incidents[keep].copy()
cleaned["EngineAttended"] = incidents["EngineAttended"]
cleaned["EnginesInTable"] = engines_per_incident.reindex(cleaned["IncidentNumber"], fill_value=0).to_numpy()
assert cleaned["IncidentNumber"].is_unique, "one row per incident"
cleaned.to_csv("assets/data/week07/cleaned_incidents.csv", index=False)

print("Pipeline summary")
print(f"  Engine rows:        900 -> {len(engines)} matched + {len(orphans)} set aside (9 duplicates removed)")
print(f"  Incidents:          {len(cleaned)} rows, IncidentNumber unique")
print(f"  Attendance times:   {cleaned['FirstPumpArriving_AttendanceTime'].isna().sum()} blank kept (no engine attended), flagged")
print(f"  DateAndTimeReturned: {mobilisations['ReturnedAt'].notna().sum()} parsed explicitly, {mobilisations['ReturnedAt'].isna().sum()} NaT")
print(f"  Districts:          12 labels -> {cleaned['IncGeo_BoroughName'].nunique()};  incident groups: 4 -> {cleaned['IncidentGroup'].nunique()}")
print("\nSaved assets/data/week07/cleaned_incidents.csv")

## Translating this to your fire-brigade tables

Apply what you've learned to your own data! In the sprint:

1. Audit the tables before anyone fixes anything. Share the audit slices across the team. Type each person's findings into your team's record with numbers (`.info()`, `.isna().sum()`, `.duplicated().sum()`, `.value_counts()`, `.describe()`, and `.isin()` both ways for the join)
2. Log one `DECISION:` entry for each problem, with what, why, and owner, *before* anyone implements it. The [Introduction](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/07/Introduction.ipynb), section "0:45–1:15: Cleaning <!-- plain-language: everyday decision -->decisions", lists the problems to log a position on
3. Make every cleaning cell print its before and after counts. Check that the counts match what the <!-- plain-language: everyday decision -->decision predicted. Ask your Associate to draft the code. Use the counts to check its work
4. Run your first `groupby` passes at the research questions on the cleaned tables. Your `data/cleaned_incidents.csv` comes out of the same notebook. Write your methodology section from the <!-- plain-language: everyday decision -->decision log

The real tables have **hundreds of thousands** of rows each. The **pipeline is the same**, and the <!-- plain-language: everyday decision -->decisions are harder, so log each one with your reasoning.

*Related pages: [Introduction](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/07/Introduction.ipynb) (the schedule) · [data note](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/lfb-data.md) · [Stoa workflow guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/stoa-workflow.md) · [group project brief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/group-project.md)*